# Closed-System TEBD for the Transverse-Field Ising Model

## Abstract

This notebook demonstrates the **Time-Evolving Block Decimation (TEBD)** algorithm applied to a closed quantum system: the Transverse-Field Ising Model (TFIM). TEBD is a tensor-network method that efficiently simulates the real-time dynamics of 1D quantum lattices by representing the quantum state as a **Matrix Product State (MPS)** and evolving it with Trotter-decomposed local gates.

### Key Ideas

- **Matrix Product State (MPS):** The many-body wavefunction $|\psi\rangle$ is written as a product of local tensors $B^{[i]}$ contracted along a virtual (bond) dimension $\chi$. Entanglement is encoded in the Schmidt values $\Lambda^{[i]}$ at each bond.

- **Suzuki-Trotter decomposition:** The time evolution operator $e^{-i H \Delta t}$ is split into a product of two-site gates using the 2nd-order decomposition:
$$
e^{-iH\Delta t} \approx \prod_{i\,\text{even}} e^{-iH_i\Delta t/2} \prod_{i\,\text{odd}} e^{-iH_i\Delta t} \prod_{i\,\text{even}} e^{-iH_i\Delta t/2} + \mathcal{O}(\Delta t^3)
$$

- **Bond update:** Each two-site gate is applied to a pair of neighboring tensors. The updated bond is re-compressed via SVD truncation, keeping only the largest $\chi_{\max}$ Schmidt values.

This notebook is self-contained and uses the helper modules `a_mps.py`, `b_model.py`, and `TFIM_tebd.py`.

## The Transverse-Field Ising Hamiltonian (Closed System)

We study the 1D TFIM on $L$ sites with **open (finite) boundary conditions**:

$$
H = -J \sum_{i=0}^{L-2} \sigma^x_i \sigma^x_{i+1} + B \sum_{i=0}^{L-1} \sigma^z_i
$$

where:
- $J > 0$ is the ferromagnetic Ising coupling between nearest-neighbor spins,
- $B$ is the transverse magnetic field strength,
- $\sigma^x, \sigma^z$ are Pauli matrices.

The competition between $J$ (which favors alignment along $x$) and $B$ (which favors alignment along $z$) drives a quantum phase transition at $B/J = 1$.

### Bond Hamiltonian

For TEBD, the Hamiltonian is decomposed into two-site bond terms $H = \sum_i h_{i,i+1}$ with:
$$
h_{i,i+1} = -J\, \sigma^x_i \sigma^x_{i+1} + w_L B\, \sigma^z_i \otimes I + w_R B\, I \otimes \sigma^z_{i+1}
$$
where the weights $w_L, w_R \in \{0.5, 1.0\}$ ensure that single-site terms are counted exactly once across all bonds.

In [ ]:
import numpy as np
from numpy import random
from scipy.linalg import svd, expm
import matplotlib.pyplot as plt
import matplotlib as mpl

from a_mps import SimpleMPS, init_FM_MPS, split_truncate_theta
from b_model import TFIModel

## MPS Structure: The `SimpleMPS` Class

The quantum state $|\psi\rangle$ is represented as an MPS in **right-canonical form**:

$$
|\psi\rangle = \sum_{\sigma_0,\ldots,\sigma_{L-1}} \Lambda^{[0]} B^{[0]}_{\sigma_0} \Lambda^{[1]} B^{[1]}_{\sigma_1} \cdots B^{[L-1]}_{\sigma_{L-1}} |\sigma_0 \cdots \sigma_{L-1}\rangle
$$

Each tensor $B^{[i]}$ has three legs:
- **Left virtual** leg of dimension $\chi_L$ (bond $i$)
- **Physical** leg of dimension $d$ (local Hilbert space; $d=2$ for spin-$1/2$)
- **Right virtual** leg of dimension $\chi_R$ (bond $i+1$)

The Schmidt values $\Lambda^{[i]}$ (stored as `Ss[i]`) at bond $i$ quantify the bipartite entanglement between sites $[0,i-1]$ and $[i, L-1]$. The von Neumann entanglement entropy is:
$$
S_i = -\sum_\alpha \lambda_\alpha^2 \ln \lambda_\alpha^2
$$

The effective **two-site wavefunction** on sites $i$ and $j=i+1$ in mixed canonical form is obtained as:
$$
\Theta^{[ij]}_{v_L, \sigma_i, \sigma_j, v_R} = \Lambda^{[i]} B^{[i]}_{\sigma_i} B^{[j]}_{\sigma_j}
$$

This is the object that TEBD gates act on.

In [ ]:
# Key methods of SimpleMPS (from a_mps.py)

# SimpleMPS.__init__: stores Bs (list of rank-3 tensors), Ss (Schmidt values), and bc

# get_theta1(i): effective single-site wavefunction on site i in mixed canonical form
#   theta[vL, i, vR] = diag(Ss[i]) @ Bs[i]
#   Implementation:
#   return np.tensordot(np.diag(self.Ss[i]), self.Bs[i], [1, 0])

# get_theta2(i): effective two-site wavefunction on sites i, j=i+1
#   theta[vL, i, j, vR] = get_theta1(i) @ Bs[j]
#   Implementation:
#   j = (i + 1) % self.L
#   return np.tensordot(self.get_theta1(i), self.Bs[j], [2, 0])

# entanglement_entropy(): von Neumann entropy S = -sum(s^2 * log(s^2)) for each bond
#   Implementation:
#   result = []
#   for i in bonds:
#       S = self.Ss[i].copy()
#       S[S < 1.e-20] = 0.
#       S2 = S * S
#       result.append(-np.sum(S2 * np.log(S2)))
#   return np.array(result)

# Demonstrate the MPS structure with a small example
L_demo = 4
psi_demo = init_FM_MPS(L=L_demo, d=2, bc='finite')
print("Number of sites:", psi_demo.L)
print("Boundary conditions:", psi_demo.bc)
print("Bond dimensions (chi):", psi_demo.get_chi())
print("Shape of B[0]:", psi_demo.Bs[0].shape, "  (chi_L, d, chi_R)")
print("Schmidt values at bond 1:", psi_demo.Ss[1])
print("Entanglement entropy (bonds 1..L-1):", psi_demo.entanglement_entropy())

## TEBD Algorithm: 2nd-Order Trotter Decomposition

### Step 1 — Exponentiate bond Hamiltonians

Each two-site gate is obtained by exponentiating the local bond Hamiltonian:
$$
U^{[i]} = e^{-i h_{i,i+1} \Delta t}
$$
For the **2nd-order Suzuki-Trotter** scheme, even-indexed bonds get a half-step gate $e^{-i h_{\text{even}} \Delta t/2}$ and odd-indexed bonds get a full-step gate $e^{-i h_{\text{odd}} \Delta t}$. Each TEBD step applies the sequence **[even, odd, even]**.

### Step 2 — Apply gate and update bond

For each bond $(i, j)$:
1. Build $\Theta = \Lambda^{[i]} B^{[i]} B^{[j]}$  (shape `vL i j vR`)
2. Contract: $\tilde{\Theta} = U^{[i]} \Theta$
3. Reshape $\tilde{\Theta}$ to matrix `(vL*i, j*vR)` and perform SVD:
   $$\tilde{\Theta} = X \cdot \text{diag}(S) \cdot Z$$
4. Truncate: keep only the $\chi_{\max}$ largest singular values
5. Renormalize $S \to S / \|S\|$ and update:
   - $G^{[i]} = (\Lambda^{[i]})^{-1} X$  (left-canonical)
   - $B^{[i]} = G^{[i]} \cdot \text{diag}(S)$  (absorb new Schmidt values)
   - $\Lambda^{[j]} = S$
   - $B^{[j]} = Z$  (right-canonical)

The truncation error is $\varepsilon = 1 - \|S_{\text{kept}}\|^2 / \|S_{\text{full}}\|^2$.

In [ ]:
# calc_U_bonds: exponentiate bond Hamiltonians into TEBD gates (from TFIM_tebd.py)

def calc_U_bonds(H_bonds, dt, L):
    """Given H_bonds, compute U_bonds[i] = expm(-i*dt*H_bonds[i]).

    Uses 2nd-order Suzuki-Trotter:
      - Even bonds: dt/2 step  ->  e^{-i H_even dt/2}
      - Odd  bonds: dt   step  ->  e^{-i H_odd  dt  }
    Applied in sequence [even, odd, even] per TEBD step.

    Each U has legs (i out, (i+1) out, i in, (i+1) in), i.e. shape (d, d, d, d).
    """
    d = H_bonds[0].shape[0]
    U_bonds = {}
    for i in range(L - 1):
        H = np.reshape(H_bonds[i], [d * d, d * d])
        if i % 2 == 0:
            U = expm(-1j * dt / 2 * H)
            U_bonds[i] = np.reshape(U, [d, d, d, d])
        else:
            U = expm(-1j * dt * H)
            U_bonds[i] = np.reshape(U, [d, d, d, d])
    return U_bonds

In [ ]:
# update_bond and run_TEBD (from TFIM_tebd.py)

TruncE = []

def update_bond(psi, i, U_bond, chi_max, eps):
    """Apply U_bond acting on sites i and j=i+1 to psi, then truncate."""
    j = (i + 1) % psi.L
    # Build two-site wavefunction in mixed canonical form
    theta = psi.get_theta2(i)   # vL i j vR
    # Apply two-site gate
    Utheta = np.tensordot(U_bond, theta, axes=([2, 3], [1, 2]))  # i j [i*] [j*], vL [i] [j] vR
    Utheta = np.transpose(Utheta, [2, 0, 1, 3])   # vL i j vR
    # SVD split and truncate
    Ai, Sj, Bj, trunc = split_truncate_theta(Utheta, chi_max, eps)
    TruncE.append(trunc)
    # Update MPS tensors
    Gi = np.tensordot(np.diag(psi.Ss[i] ** (-1)), Ai, axes=[1, 0])  # vL [vL*], [vL] i vC
    psi.Bs[i] = np.tensordot(Gi, np.diag(Sj), axes=[2, 0])          # vL i [vC], [vC] vC
    psi.Ss[j] = Sj
    psi.Bs[j] = Bj


def run_TEBD(psi, U_bonds, N_steps, chi_max, eps):
    """Evolve psi for N_steps using 2nd-order Trotter [even, odd, even]."""
    Nbonds = psi.L - 1 if psi.bc == 'finite' else psi.L
    assert len(U_bonds) == Nbonds
    for n in range(N_steps):
        for k in [0, 1, 0]:   # even, odd, even
            for i_bond in range(k, Nbonds, 2):
                update_bond(psi, i_bond, U_bonds[i_bond], chi_max, eps)

## Running the Simulation

We simulate real-time dynamics of the TFIM starting from the ferromagnetic product state $|\psi_0\rangle = |+x\rangle^{\otimes L}$ (all spins polarized along $+x$).

Parameters:
- $L = 6$ sites
- $J = 1.0$, $B = 1.0$ (critical point)
- $t_{\max} = 2.0$, $\Delta t = 0.01$
- $\chi_{\max} = 100$, SVD cutoff $\varepsilon = 10^{-8}$

We track:
- $\langle \sigma^z_i \rangle(t)$ and $\langle \sigma^x_i \rangle(t)$ (mean magnetization)
- Entanglement entropy $S_i(t)$ at each bond
- Accumulated truncation error

In [ ]:
# -----------------------------------------------------------------------
# Simulation parameters
# -----------------------------------------------------------------------
L    = 6
J    = 1.0
Bi   = 1.0   # initial (and fixed) transverse field
Bf   = 1.0   # final transverse field (same as Bi => constant field)
tmax = 2.0
dt   = 0.01

chi_max = 100
eps_svd = 1e-8

Nsteps = int(tmax / dt)
tspan  = np.linspace(0.0, tmax, Nsteps)

# Build model and initial state
M   = TFIModel(L=L, J=J, Bi=Bi, Bf=Bf, time=0.0, tf=tmax, bc='finite')
psi = init_FM_MPS(M.L, M.d, M.bc)  # |+x>^L

# Storage
S_ent_t  = []   # entanglement entropy
Sz_t     = []   # <sigma^z> per site
Sx_t     = []   # <sigma^x> per site
TruncE   = []   # truncation factors

# -----------------------------------------------------------------------
# Time evolution loop
# -----------------------------------------------------------------------
for n in range(Nsteps):
    # Rebuild model at current time (field is constant here, but kept general)
    M_t     = TFIModel(L=L, J=J, Bi=Bi, Bf=Bf, time=tspan[n], tf=tmax, bc='finite')
    U_bonds = calc_U_bonds(M_t.H_bonds, dt, L)

    run_TEBD(psi, U_bonds, N_steps=1, chi_max=chi_max, eps=eps_svd)

    S_ent_t.append(psi.entanglement_entropy())
    Sz_t.append(psi.site_expectation_value(M_t.sz))
    Sx_t.append(psi.site_expectation_value(M_t.sx))

S_ent_t = np.array(S_ent_t)   # (Nsteps, L-1)
Sz_t    = np.array(Sz_t)      # (Nsteps, L)
Sx_t    = np.array(Sx_t)      # (Nsteps, L)

print("Simulation complete.")
print("Final bond dimensions:", psi.get_chi())

# -----------------------------------------------------------------------
# Plots
# -----------------------------------------------------------------------
mpl.rcParams.update(mpl.rcParamsDefault)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: Mean magnetization
ax = axes[0]
ax.plot(tspan, np.mean(Sz_t, axis=1), label=r'$\langle\sigma^z\rangle$', color='#1090BF')
ax.plot(tspan, np.mean(Sx_t, axis=1), label=r'$\langle\sigma^x\rangle$', color='#DD5400')
ax.set_xlabel('Time $t$')
ax.set_ylabel('Mean magnetization')
ax.set_title('Magnetization dynamics')
ax.legend()

# Panel 2: Entanglement entropy per bond
ax = axes[1]
for b in range(L - 1):
    ax.plot(tspan, S_ent_t[:, b], label=f'bond {b}')
ax.set_xlabel('Time $t$')
ax.set_ylabel('Entanglement entropy $S$')
ax.set_title('von Neumann entropy per bond')
ax.legend(fontsize=8)

# Panel 3: Bond dimensions over time (sampled)
sample_steps = np.linspace(0, Nsteps - 1, 10, dtype=int)
ax = axes[2]
ax.plot(tspan, S_ent_t[:, (L - 1) // 2], color='#8500D1', lw=1.5)
ax.set_xlabel('Time $t$')
ax.set_ylabel('Entanglement entropy')
ax.set_title(f'Central bond entanglement (bond {(L-1)//2})')

plt.tight_layout()
plt.show()

## Results and Discussion

### Magnetization Dynamics

Starting from the $|{+x}\rangle^{\otimes L}$ state (an eigenstate of $\sigma^x$), the transverse field $B\sigma^z$ drives precession of each spin away from the $x$-axis. The ferromagnetic coupling $-J\sigma^x\sigma^x$ tends to maintain $x$-alignment. At the critical point $B=J=1$, the competition produces non-trivial oscillatory dynamics.

- $\langle\sigma^x\rangle$ starts at 1 (all spins aligned) and decreases as the state loses $x$-polarization.
- $\langle\sigma^z\rangle$ starts near 0 and oscillates as the field generates $z$-magnetization.

### Entanglement Growth

The entanglement entropy $S_i(t)$ grows in time as quantum correlations spread across the chain. For a quantum quench in the TFIM, the entropy is expected to grow **linearly** at early times (light-cone spreading of entanglement) before saturating for a finite system.

The central bond carries the most entanglement, reflecting the fact that it separates the largest equal-sized bipartitions.

### Truncation and Accuracy

The MPS bond dimension $\chi$ grows as entanglement builds up. The truncation error per step is controlled by `eps_svd`. With $\chi_{\max} = 100$ and $\varepsilon = 10^{-8}$, the simulation is essentially exact for $L=6$ (the maximum required $\chi$ is small for short chains).

### Scaling

The computational cost of TEBD scales as $\mathcal{O}(\chi^3 d^3 L / \Delta t)$, making it feasible for systems where entanglement is moderate (i.e., area-law states or early-time dynamics before volume-law entanglement builds up).